In [7]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

from src.models.gru import GRU
from src.data_loader import load_symbol
from src.features import build_features

SEED = 42
L = 30
torch.manual_seed(SEED)
np.random.seed(SEED)

In [8]:
model = GRU(input_size=8, hidden_size=16, output_size=1)
print(model(torch.randn(32, 20, 8)).shape)

torch.Size([32, 1])


In [9]:
es_df = load_symbol('ES=F', start='2000-01-01', end='2022-01-01')
es_features = build_features(es_df)

es_features['y'] = es_features['log_return'].shift(-1)
es_features = es_features.replace([np.inf, -np.inf], np.nan).dropna()

y = es_features['y']
X = es_features.drop(columns=['y'])
print(X.shape, y.shape)

(5355, 16) (5355,)


/Users/sergheibigboy/cpo_rnn/.venv/lib/python3.13/site-packages/yfinance/utils.py:568: Pandas4Warning: The inplace keyword in DataFrame.sort_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quotes.sort_index(inplace=True)
/Users/sergheibigboy/cpo_rnn/.venv/lib/python3.13/site-packages/yfinance/utils.py:514: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(
/Users/sergheibigboy/cpo_rnn/.venv/lib/python3.13/site-packages/yfinance/utils.py:518: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={
/Users/sergheibi

In [10]:
n = len(X)
i_val = int(n * 0.60)
i_test = int(n * 0.85)

X_train, X_val, X_test = X.iloc[:i_val], X.iloc[i_val:i_test], X.iloc[i_test:]
y_train, y_val, y_test = y.iloc[:i_val], y.iloc[i_val:i_test], y.iloc[i_test:]

sx = StandardScaler().fit(X_train)
sy = StandardScaler().fit(y_train.values.reshape(-1, 1))

def make_windows(X_part, y_part, L):
    Xs = sx.transform(X_part)
    ys = sy.transform(y_part.values.reshape(-1, 1)).ravel()
    xw, yw = [], []
    for i in range(L, len(Xs) + 1):
        xw.append(Xs[i - L:i])
        yw.append(ys[i - 1])
    xw = torch.tensor(np.array(xw), dtype=torch.float32)
    yw = torch.tensor(np.array(yw), dtype=torch.float32).unsqueeze(1)
    return xw, yw

Xtr, ytr = make_windows(X_train, y_train, L)
Xva, yva = make_windows(X_val, y_val, L)
Xte, yte = make_windows(X_test, y_test, L)
print(Xtr.shape, Xva.shape, Xte.shape)

torch.Size([3184, 30, 16]) torch.Size([1309, 30, 16]) torch.Size([775, 30, 16])


In [11]:
model = GRU(input_size=Xtr.shape[2], hidden_size=32, output_size=1)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.MSELoss()
loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=64, shuffle=True)

best_val = float('inf')
best_state = None
patience, bad = 8, 0

for epoch in range(100):
    model.train()
    tr_loss = 0.0
    for xb, yb in loader:
        opt.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tr_loss += loss.item() * len(xb)
    tr_loss /= len(Xtr)

    model.eval()
    with torch.no_grad():
        val_loss = loss_fn(model(Xva), yva).item()

    print(f'{epoch:3d} train {tr_loss:.4f} val {val_loss:.4f}')

    if val_loss < best_val:
        best_val = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        bad = 0
    else:
        bad += 1
        if bad >= patience:
            break

model.load_state_dict(best_state)

  0 train 1.0174 val 0.3698
  1 train 1.0021 val 0.3848
  2 train 0.9976 val 0.3692
  3 train 0.9925 val 0.3641
  4 train 0.9882 val 0.3929
  5 train 0.9866 val 0.3661
  6 train 0.9827 val 0.4208
  7 train 0.9811 val 0.3805
  8 train 0.9787 val 0.3912
  9 train 0.9748 val 0.4253
 10 train 0.9726 val 0.3966
 11 train 0.9681 val 0.4134


<All keys matched successfully>

In [12]:
def evaluate(Xs, ys, name):
    model.eval()
    with torch.no_grad():
        p = model(Xs).numpy().ravel()
    t = ys.numpy().ravel()
    mse = np.mean((p - t) ** 2)
    mse_zero = np.mean(t ** 2)
    ic = np.corrcoef(p, t)[0, 1] if p.std() > 0 else float('nan')
    hit = np.mean(np.sign(p) == np.sign(t))
    print(f'{name}: mse {mse:.4f} | baseline(0) {mse_zero:.4f} | IC {ic:.4f} | hit {hit:.4f} | pred std {p.std():.4f}')
    return p, t

_ = evaluate(Xtr, ytr, 'train')
_ = evaluate(Xva, yva, 'val  ')
p_te, t_te = evaluate(Xte, yte, 'test ')

train: mse 0.9846 | baseline(0) 0.9998 | IC 0.1301 | hit 0.5367 | pred std 0.0895
val  : mse 0.3641 | baseline(0) 0.3500 | IC 0.0161 | hit 0.4683 | pred std 0.0671
test : mse 1.1813 | baseline(0) 1.1429 | IC 0.0923 | hit 0.4426 | pred std 0.0540
